# Multimodal Cooking Assistant — EPIC-KITCHENS (30 Classes) -> YOLOv5s -> Hailo-8 ONNX

Upload this notebook to **Google Colab** and select a CUDA GPU (**L4**, **A100**, or **T4**).

### v2 Accuracy & Class Upgrades
1. **30 EPIC-KITCHENS-aligned classes** — separates specific tools & ingredients (`onion`, `tomato`, `potato`, `carrot`, `garlic`, `pepper`, `egg`, `cheese`, `bread`, `meat`, `lid`, `hob`, `tap`, `sink`, `jar`, `fork`, etc.) so your DAG & recipe checker know exact items without sacrificing Hailo-8 speed.
2. **2 FPS extraction (up to 300 frames/video)** split **80% train / 20% val by video** (zero session leakage).
3. **Higher-precision auto-labeling** using **`yolov8l-worldv2.pt`** (Large) + tiny-box noise filtering (`w, h >= 1.5%`) + per-class count summary.
4. **Better YOLOv5s convergence** (`100` epochs, `--cos-lr`, `--patience 20`) + saves training curves (`results.png`, `confusion_matrix.png`, `PR_curve.png`, `results.csv`) to Google Drive alongside `best.pt` and `kitchen_yolov5s.onnx`.

In [ ]:
# 1. Verify Colab CUDA GPU
import torch
!nvidia-smi
assert torch.cuda.is_available(), "CUDA GPU not active! Go to Runtime -> Change runtime type -> L4/T4 GPU."
print(f"PyTorch {torch.__version__} | Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone --depth 1 https://github.com/ultralytics/yolov5.git
!pip install -q -r /content/yolov5/requirements.txt ultralytics onnx onnxslim onnxruntime onnxscript

In [ ]:
# 3. Extract 2 FPS frames from EPIC-KITCHENS .MP4s (split by video: 80% train / 20% val)
import cv2
from pathlib import Path

VIDEO_ROOT = Path('/content/drive/MyDrive/EPIC_Kitchens')
DATA_ROOT  = Path('/content/yolo_dataset')
DRIVE_OUT  = VIDEO_ROOT / 'yolo_outputs_30cls'

TARGET_FPS = 2.0
MAX_FRAMES_PER_VIDEO = 300  # Set None to extract every sampled frame

videos = sorted(VIDEO_ROOT.rglob('*.[mM][pP]4'))
print(f'Found {len(videos)} MP4 video(s) under {VIDEO_ROOT}')
assert len(videos) > 0, f'No .mp4/.MP4 videos found in {VIDEO_ROOT}'

split_idx = max(1, int(len(videos) * 0.8))
splits = {
    'train': videos[:split_idx],
    'val': videos[split_idx:] if len(videos) > 1 else videos[-1:],
}

for split_name, vid_list in splits.items():
    img_dir = DATA_ROOT / 'images' / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    for vid_path in vid_list:
        cap = cv2.VideoCapture(str(vid_path))
        orig_fps = cap.get(cv2.CAP_PROP_FPS) or 50.0
        step = max(1, int(round(orig_fps / TARGET_FPS)))
        frame_idx, saved = 0, 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % step == 0:
                out_file = img_dir / f'{vid_path.stem}_f{frame_idx:06d}.jpg'
                cv2.imwrite(str(out_file), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
                saved += 1
                if MAX_FRAMES_PER_VIDEO and saved >= MAX_FRAMES_PER_VIDEO:
                    break
            frame_idx += 1
        cap.release()
        print(f'[{split_name}] {vid_path.name}: saved {saved} frames')

In [ ]:
# 4. Auto-label with YOLO-World Large (yolov8l-worldv2.pt) -> 30 EPIC-KITCHENS classes
from collections import Counter
from pathlib import Path
from ultralytics import YOLO

CLASS_NAMES = {
    0: 'hand', 1: 'pan', 2: 'pot', 3: 'lid', 4: 'hob', 5: 'tap', 6: 'sink',
    7: 'knife', 8: 'spatula', 9: 'spoon', 10: 'fork', 11: 'cutting_board',
    12: 'bowl', 13: 'plate', 14: 'cup', 15: 'bottle', 16: 'jar', 17: 'appliance',
    18: 'onion', 19: 'tomato', 20: 'potato', 21: 'carrot', 22: 'garlic', 23: 'pepper',
    24: 'egg', 25: 'cheese', 26: 'bread', 27: 'meat', 28: 'vegetable', 29: 'food',
}

PROMPT_TO_TARGET_ID = {
    'hand': 0,
    'pan': 1, 'frying pan': 1, 'skillet': 1, 'wok': 1,
    'pot': 2, 'saucepan': 2,
    'pot lid': 3, 'pan lid': 3,
    'stove': 4, 'stove burner': 4, 'cooktop': 4,
    'faucet': 5, 'water tap': 5,
    'kitchen sink': 6,
    'knife': 7, 'kitchen knife': 7,
    'spatula': 8, 'turner': 8,
    'spoon': 9, 'wooden spoon': 9, 'ladle': 9,
    'fork': 10,
    'cutting board': 11, 'chopping board': 11,
    'bowl': 12,
    'plate': 13,
    'cup': 14, 'mug': 14, 'drinking glass': 14,
    'bottle': 15,
    'jar': 16,
    'microwave': 17, 'kettle': 17, 'blender': 17, 'toaster': 17, 'coffee maker': 17,
    'onion': 18,
    'tomato': 19,
    'potato': 20,
    'carrot': 21,
    'garlic': 22,
    'bell pepper': 23, 'chili pepper': 23,
    'egg': 24,
    'cheese': 25,
    'bread': 26, 'tortilla': 26, 'toast': 26,
    'raw meat': 27, 'chicken': 27, 'sausage': 27, 'bacon': 27, 'fish': 27,
    'broccoli': 28, 'lettuce': 28, 'cucumber': 28, 'mushroom': 28, 'spinach': 28, 'vegetable': 28,
    'cooked food': 29, 'pasta': 29, 'rice': 29, 'salad': 29, 'butter': 29,
}
PROMPTS = list(PROMPT_TO_TARGET_ID.keys())

labeler = YOLO('yolov8l-worldv2.pt')
labeler.set_classes(PROMPTS)

for split_name in ('train', 'val'):
    img_dir = DATA_ROOT / 'images' / split_name
    lbl_dir = DATA_ROOT / 'labels' / split_name
    lbl_dir.mkdir(parents=True, exist_ok=True)

    results = labeler.predict(
        source=str(img_dir),
        conf=0.28,
        iou=0.45,
        max_det=25,
        stream=True,
        verbose=False,
    )
    labeled_imgs, total_boxes = 0, 0
    counts = Counter()
    for res in results:
        img_path = Path(res.path)
        lines = []
        if res.boxes is not None and len(res.boxes) > 0:
            cls_ids = res.boxes.cls.cpu().numpy().astype(int)
            xywhn = res.boxes.xywhn.cpu().numpy()
            for cid, (cx, cy, w, h) in zip(cls_ids, xywhn):
                if w < 0.015 or h < 0.015:  # Skip tiny noisy detections
                    continue
                target_id = PROMPT_TO_TARGET_ID[PROMPTS[cid]]
                lines.append(f'{target_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
                counts[target_id] += 1
        if lines:
            labeled_imgs += 1
            total_boxes += len(lines)
            (lbl_dir / f'{img_path.stem}.txt').write_text('\n'.join(lines) + '\n')
    print(f'\n[{split_name}] Auto-labeled {labeled_imgs} images ({total_boxes} boxes)')
    print('  Class counts:', {CLASS_NAMES[k]: v for k, v in sorted(counts.items())})

In [ ]:
# 5. Spot-check auto-labeled images before training
import random, cv2
import matplotlib.pyplot as plt

COLORS = {i: ((37 * i + 80) % 200 + 55, (97 * i + 140) % 200 + 55, (157 * i + 40) % 200 + 55) for i in CLASS_NAMES}

sample_lbls = sorted((DATA_ROOT / 'labels' / 'train').glob('*.txt'))
for lbl_path in random.sample(sample_lbls, min(4, len(sample_lbls))):
    img = cv2.cvtColor(cv2.imread(str(DATA_ROOT / 'images' / 'train' / f'{lbl_path.stem}.jpg')), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for line in lbl_path.read_text().splitlines():
        cid, cx, cy, bw, bh = map(float, line.split())
        cid = int(cid)
        x1, y1 = int((cx - bw / 2) * W), int((cy - bh / 2) * H)
        x2, y2 = int((cx + bw / 2) * W), int((cy + bh / 2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[cid], 2)
        cv2.putText(img, CLASS_NAMES[cid], (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, COLORS[cid], 2)
    plt.figure(figsize=(7, 5))
    plt.imshow(img)
    plt.title(lbl_path.stem)
    plt.axis('off')
    plt.show()

In [ ]:
# 6. Write kitchen_data.yaml and train YOLOv5s (640x640, 100 epochs, cos-lr, patience 20)
import yaml

data_cfg = {
    'path': str(DATA_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': CLASS_NAMES,
}
yaml_path = Path('/content/kitchen_data.yaml')
yaml_path.write_text(yaml.safe_dump(data_cfg, sort_keys=False))

%cd /content/yolov5
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --patience 20 \
  --cos-lr \
  --data /content/kitchen_data.yaml \
  --weights yolov5s.pt \
  --project /content/runs/train \
  --name kitchen_yolov5s \
  --device 0 \
  --cache \
  --exist-ok

In [ ]:
# 7. Validate & display training curves + confusion matrix
from IPython.display import Image, display

run_dir = Path('/content/runs/train/kitchen_yolov5s')
best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'

!python /content/yolov5/val.py --data /content/kitchen_data.yaml --weights "{best_pt}" --img 640 --device 0

for chart in ('results.png', 'confusion_matrix.png', 'PR_curve.png'):
    p = run_dir / chart
    if p.exists():
        display(Image(filename=str(p)))

In [ ]:
# 8. Export best.pt -> Hailo-8-ready static ONNX (opset 12, simplified) and verify
import shutil, numpy as np, onnx, onnxruntime as ort

!python /content/yolov5/export.py \
  --weights "{best_pt}" \
  --imgsz 640 640 \
  --batch-size 1 \
  --include onnx \
  --opset 12 \
  --simplify

final_onnx = Path('/content/kitchen_yolov5s.onnx')
shutil.copy2(best_pt.with_suffix('.onnx'), final_onnx)

onnx.checker.check_model(onnx.load(str(final_onnx)))
sess = ort.InferenceSession(str(final_onnx), providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
out = sess.run(None, {inp.name: np.zeros((1, 3, 640, 640), dtype=np.float32)})[0]
print(f'Verified ONNX: input={inp.name} {inp.shape} -> output={out.shape}')

In [ ]:
# 9. Save weights, ONNX, and all training charts/metrics back to Google Drive + download
from google.colab import files
import shutil

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, DRIVE_OUT / 'kitchen_yolov5s_best.pt')
shutil.copy2(final_onnx, DRIVE_OUT / 'kitchen_yolov5s.onnx')
for fname in ('results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'results.csv'):
    src = run_dir / fname
    if src.exists():
        shutil.copy2(src, DRIVE_OUT / fname)
print(f'Saved weights, ONNX, and charts to Google Drive: {DRIVE_OUT}')

files.download(str(final_onnx))
files.download(str(best_pt))